# ECON 4370 / BANA 4373 — Homework 4 
## Difference-in-Differences: From Intuition to Estimation

**Posted:** Mar. 9  
**Due:** Mar. 23 (11:59 PM, Blackboard Ultra)

> **How to use this notebook:**
> - Replace every **TODO** with your answer (code or written response).
> - Keep written responses in **Markdown** cells. Keep code in **code** cells.
> - The notebook must run **top → bottom without errors** before you submit.
> - Save as `.ipynb` and upload to Blackboard Ultra.

---

### Context

You cannot always run a randomized experiment. When treatment is assigned by policy, geography, or circumstance rather than by a coin flip, **Difference-in-Differences (DiD)** is one of the most widely used tools for recovering a causal estimate.

In this homework you will:
1. Reconstruct the logic of DiD by hand from a 2×2 table.
2. Simulate calibrated panel data with a **known** treatment effect.
3. Estimate the DiD estimator both manually and via OLS regression.
4. Visualize the parallel trends assumption.
5. **Break the estimator** — violate parallel trends and observe the bias.
6. Apply the framework to a real policy context.

## Student Information
- **Name:** TODO
- **Section / Time:** TODO
- **NetID / Email:** TODO (optional)

---
## 0) Setup — Imports and Folder Structure

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import os

# Reproducibility
np.random.seed(4373)

# Folder structure (mirrors course convention)
for folder in ["data_raw", "data_clean", "exports"]:
    os.makedirs(folder, exist_ok=True)

print("Setup complete.")

---
# Part 1: The DiD Logic (No Code)

Answer the following questions in your own words. Keep each answer concise (2–4 sentences unless otherwise noted).

## Question 1 — The Fundamental Problem

Briefly explain why we cannot simply compare outcomes for treated units before and after a policy to measure its causal effect. What is the key counterfactual we are missing?

**Answer:** TODO

## Question 2 — The 2×2 Table

The table below shows average employment rates (in percentage points) for two groups of counties before and after a minimum wage increase was introduced in the **treatment** group.

| | **Before** | **After** |
|---|---|---|
| **Treatment counties** | 62.0 | 65.5 |
| **Control counties** | 60.0 | 61.5 |

**(a)** What is the naïve before–after estimate for the treatment group?

**(b)** What is the control group's change over the same period?

**(c)** What is the DiD estimate? Show your arithmetic.

**(d)** Interpret the DiD estimate in a plain-English sentence. What does this number mean?

**Answer (a):** TODO

**Answer (b):** TODO

**Answer (c):** TODO

**Answer (d):** TODO

## Question 3 — Parallel Trends

**(a)** State the **parallel trends assumption** in your own words (1–2 sentences).

**(b)** Why can we **never fully verify** parallel trends using data alone?

**(c)** What visual evidence would make you *more* confident the assumption holds?

**Answer (a):** TODO

**Answer (b):** TODO

**Answer (c):** TODO

## Question 4 — Selection Bias Link

Recall the **SDO decomposition** from the potential outcomes lecture:

$$\text{SDO} = \text{ATE} + \underbrace{E[Y_i(0)|D_i=1] - E[Y_i(0)|D_i=0]}_{\text{Selection Bias}}$$

Explain in 2–3 sentences how DiD attempts to remove the selection bias term. What is it using the control group's trend to do?

**Answer:** TODO

---
# Part 2: Simulating Panel Data with a Known Treatment Effect

You will now generate a **calibrated synthetic dataset** where we know the true causal effect. This lets you check whether your estimator recovers it.

### Context

Imagine a state job-training program rolled out in **Year 2** for a subset of counties. You observe each county in **Year 1** (pre-policy) and **Year 2** (post-policy). The outcome is the county-level employment rate (percentage points).

**Design parameters (do not change):**
- `N_counties = 200` — 100 treatment, 100 control
- `TRUE_ATT = 4.0` — the true causal effect (pp) you want to recover
- Treatment counties have a **higher baseline** employment rate (selection bias built in)
- Both groups share the **same time trend** (parallel trends holds by construction)

## Question 5 — Write Your Prior

**Before running any code**, answer the following:

**(a)** If you ignored the control group and only compared treatment counties before vs. after, would you expect to *over-estimate*, *under-estimate*, or *correctly estimate* the true effect of 4.0 pp? Why?

**(b)** If you used a cross-sectional comparison (treatment vs. control *only in Year 2*), would selection bias push your estimate up or down relative to the truth? Why?

**Prior (a):** TODO

**Prior (b):** TODO

## Question 6 — Simulate the Data

In [ ]:
# ── Design parameters ──────────────────────────────────────────────────────
N_treat   = 100
N_control = 100
TRUE_ATT  = 4.0          # pp — the number we want to recover
COMMON_TREND = 2.0       # pp — shared time trend (same for both groups)
SELECTION_BIAS = 5.0     # pp — treatment counties start higher at baseline
NOISE_SD  = 3.0          # pp — county-level noise

# ── Build long-format panel ─────────────────────────────────────────────────
county_id  = list(range(N_treat + N_control))
treated    = [1] * N_treat + [0] * N_control

# Baseline (Year 1) employment rates
baseline_treat   = 62.0 + SELECTION_BIAS
baseline_control = 62.0

# County fixed effects (stable characteristic that shifts each county's baseline)
county_fe = np.concatenate([
    np.random.normal(baseline_treat,   NOISE_SD, N_treat),
    np.random.normal(baseline_control, NOISE_SD, N_control)
])

rows = []
for cid, treat, fe in zip(county_id, treated, county_fe):
    for year in [1, 2]:
        post = 1 if year == 2 else 0
        treatment_effect = TRUE_ATT if (post == 1 and treat == 1) else 0.0
        noise = np.random.normal(0, NOISE_SD)
        emp_rate = fe + COMMON_TREND * post + treatment_effect + noise
        rows.append({"county": cid, "year": year, "post": post,
                     "treated": treat, "emp_rate": emp_rate})

df = pd.DataFrame(rows)

# Save raw data
df.to_csv("data_raw/panel_simulated.csv", index=False)
print(f"Dataset shape: {df.shape}")
df.head(8)

**Briefly describe the dataset (2–3 sentences):** What does each row represent? How many rows are there total, and why?

**Answer:** TODO

---
# Part 3: Naïve Estimates vs. DiD (Manual Calculation)

## Question 7 — Reconstruct the 2×2 Table

**Clarifying prompt (so you do not get stuck on pandas):**

You are trying to recover the four cell means in the 2×2 DiD table. A very direct way is to group by `treated` and `post`, then take the mean of `emp_rate`.

**Hint (you may use this directly):**
```python
df.groupby(["treated", "post"])["emp_rate"].mean()
```

Once you see the four means, store them in:
`treat_pre`, `treat_post`, `ctrl_pre`, `ctrl_post`.

In [ ]:
# TODO: Compute the mean employment rate for each of the 4 cells:
#   (treated=1, post=0), (treated=1, post=1),
#   (treated=0, post=0), (treated=0, post=1)
# Store results in variables named:
#   treat_pre, treat_post, ctrl_pre, ctrl_post

# YOUR CODE HERE


print(f"Treated   | Before: {treat_pre:.2f}  After: {treat_post:.2f}")
print(f"Control   | Before: {ctrl_pre:.2f}  After: {ctrl_post:.2f}")

## Question 8 — Three Estimators, Three Answers

**Clarifying prompt:**

Use the four means from Question 7. You do **not** need any new pandas work here.

Mechanically:
- Before–after for treated only = treated after − treated before
- Cross-section in post period = treated post − control post
- DiD = (treated after − treated before) − (control after − control before)

In [ ]:
# TODO: Compute the three estimators below and print them.

# 1. Naïve before–after for treated group only
naive_before_after = # TODO

# 2. Cross-sectional comparison (treated vs. control in post period only)
naive_cross_section = # TODO

# 3. Difference-in-Differences estimator
did_manual = # TODO

print(f"True ATT               : {TRUE_ATT:.2f}")
print(f"Naïve before-after     : {naive_before_after:.2f}")
print(f"Naïve cross-section    : {naive_cross_section:.2f}")
print(f"DiD (manual)           : {did_manual:.2f}")

**Interpret your results (3–4 sentences):** Which estimator comes closest to the true ATT? Are the naïve estimators biased in the direction you predicted in Question 5? Explain why each naïve estimate is off.

**Answer:** TODO

---
# Part 4: DiD via OLS Regression

The DiD estimator has an exact OLS representation. The regression we want is:

$$\text{emp\_rate}_{it} = \beta_0 + \beta_1 \cdot \text{post}_t + \beta_2 \cdot \text{treated}_i + \beta_3 \cdot (\text{post}_t \times \text{treated}_i) + \varepsilon_{it}$$

where $\hat{\beta}_3$ is the **DiD estimator**.

## Question 9 — Write Your Prior for β₃

Before running the regression, write down what you expect for $\hat{\beta}_3$ and why.

**Prior:** TODO

## Question 10 — Run the DiD Regression

**Clarifying prompt:**

In regression form, DiD is just:
\[
Y = \beta_0 + \beta_1 \text{post} + \beta_2 \text{treated} + \beta_3 (\text{post} \times \text{treated}) + u
\]

A simple way to code the interaction is:
```python
df["post_x_treated"] = df["post"] * df["treated"]
```

Then run OLS with `emp_rate` as the outcome.

In [ ]:
# TODO: Create the interaction term and run the OLS model.
# Use statsmodels formula API: smf.ols(formula, data=df).fit()
# Hint: the formula should include post, treated, and their interaction.

df["post_x_treated"] = # TODO

model = smf.ols(# TODO formula string
                , data=df).fit()

print(model.summary())

**Question 10a — Coefficient Interpretation:** In plain English, interpret each of the four coefficients ($\hat{\beta}_0$, $\hat{\beta}_1$, $\hat{\beta}_2$, $\hat{\beta}_3$). What does each one represent in the real-world context of this dataset?

**β̂₀ (intercept):** TODO

**β̂₁ (post):** TODO

**β̂₂ (treated):** TODO

**β̂₃ (post × treated):** TODO

**Question 10b:** Does $\hat{\beta}_3$ match your manual DiD calculation from Question 8? Does it match the true ATT (4.0 pp)? Why might it not be exactly 4.0?

**Answer:** TODO

---
# Part 5: Visualizing Parallel Trends

## Question 11 — The Parallel Trends Plot

**Clarifying prompt:**

First create a small summary dataset with the mean employment rate by `year` and `treated`. Then plot the two series.

**Hint (one possible approach):**
```python
means = df.groupby(["year", "treated"], as_index=False)["emp_rate"].mean()
```
Then plot one line for `treated == 1` and one line for `treated == 0`.

In [ ]:
# TODO: Create a line plot showing the mean employment rate by year
# for treated and control groups separately.
#
# Requirements:
#  - X-axis: year (1 and 2)
#  - Y-axis: mean employment rate
#  - Two lines: one for treated, one for control
#  - Add a vertical dashed line at year=1.5 labeled "Policy introduced"
#  - Label axes, add a legend, add a title
#  - Save the figure to exports/parallel_trends.png

# YOUR CODE HERE

plt.savefig("exports/parallel_trends.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved.")

**Describe your plot (2–3 sentences):** Does the visual evidence support the parallel trends assumption *before* the policy? What does the gap between the two lines in Year 2 suggest?

**Answer:** TODO

---
# Part 6: Break the Estimator — Violating Parallel Trends

> *"The best way to understand an assumption is to violate it on purpose."*

Now you will **deliberately break** the parallel trends assumption by introducing a pre-existing trend that differs across groups. This lets you see exactly how and how much the DiD estimator goes wrong.

## Question 12 — Write Your Prior Before Breaking

You are about to add an extra +3 pp trend to the treatment group *before* the policy (i.e., they were already trending upward more steeply). The true ATT remains 4.0 pp.

Before running the code: in which direction do you expect the **biased** DiD estimate to move (higher or lower than 4.0)? Why?

**Prior:** TODO

In [ ]:
# ── Simulate broken data: treatment group has a steeper pre-trend ───────────
EXTRA_TREAT_TREND = 3.0   # pp — treatment group was already rising faster

rows_broken = []
for cid, treat, fe in zip(county_id, treated, county_fe):
    for year in [1, 2]:
        post = 1 if year == 2 else 0
        treatment_effect = TRUE_ATT if (post == 1 and treat == 1) else 0.0
        # Extra trend: treatment group trends up MORE even without the policy
        extra_trend = EXTRA_TREAT_TREND * post * treat
        noise = np.random.normal(0, NOISE_SD)
        emp_rate = fe + COMMON_TREND * post + extra_trend + treatment_effect + noise
        rows_broken.append({"county": cid, "year": year, "post": post,
                             "treated": treat, "emp_rate": emp_rate})

df_broken = pd.DataFrame(rows_broken)
df_broken["post_x_treated"] = df_broken["post"] * df_broken["treated"]

# Estimate DiD on the broken data
model_broken = smf.ols("emp_rate ~ post + treated + post_x_treated", data=df_broken).fit()

biased_estimate = model_broken.params["post_x_treated"]
print(f"True ATT        : {TRUE_ATT:.2f}")
print(f"Biased estimate : {biased_estimate:.2f}")
print(f"Bias            : {biased_estimate - TRUE_ATT:.2f} pp")

**Interpret the result (3–4 sentences):** Was the bias in the direction you predicted? Explain *mechanically* why the DiD estimator picks up the extra trend as part of the treatment effect. What would an applied economist need to rule this out?

**Answer:** TODO

## Question 13 — Plot the Broken Parallel Trends

**Clarifying prompt:**

You can reuse the same plotting logic from Question 11, but do it twice:
- once using `df`
- once using `df_broken`

A simple first step is:
```python
means_orig = df.groupby(["year", "treated"], as_index=False)["emp_rate"].mean()
means_broken = df_broken.groupby(["year", "treated"], as_index=False)["emp_rate"].mean()
```
Then plot them in two side-by-side panels.

In [ ]:
# TODO: Reproduce your parallel trends plot but using df_broken.
# Side-by-side subplots (1 row, 2 cols):
#   Left panel:  original data (df) — parallel trends holds
#   Right panel: broken data (df_broken) — parallel trends violated
# Each panel should have the same axis labels, a legend, and a title.
# Save to exports/parallel_trends_comparison.png

# YOUR CODE HERE

plt.savefig("exports/parallel_trends_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

**Describe what you see (2–3 sentences):** What is the visual difference between the two panels? How does this plot demonstrate the importance of the parallel trends assumption for the validity of the DiD estimator?

**Answer:** TODO

---
# Part 7: Real-World Application — Card & Krueger (1994)

Card and Krueger (1994) used DiD to study the effect of New Jersey's 1992 minimum wage increase on fast-food employment. New Jersey raised its minimum wage from \$4.25 to \$5.05; neighboring Pennsylvania did not. The table below reports their famous "Table 3" estimates (in full-time equivalent (FTE) employees per store):

| | **Before (Feb. 1992)** | **After (Nov. 1992)** | **Δ (After − Before)** |
|---|---|---|---|
| **New Jersey (treatment)** | 20.44 | 21.03 | +0.59 |
| **Pennsylvania (control)** | 23.33 | 21.17 | −2.16 |

Source: Card & Krueger (1994), *American Economic Review*, Table 3.

## Question 14 — Replicate the DiD Estimate

In [ ]:
# TODO: Using the four values in the table above, compute
#   (a) the DiD estimate (as Card & Krueger report it)
#   (b) what NJ's employment would have been under the counterfactual
#       (i.e., if it had followed Pennsylvania's trend)

nj_before = 20.44
nj_after  = 21.03
pa_before = 23.33
pa_after  = 21.17

ck_did = # TODO
nj_counterfactual = # TODO (what NJ would have been without the min-wage hike)

print(f"Card–Krueger DiD estimate : {ck_did:.2f} FTE")
print(f"NJ counterfactual (post)  : {nj_counterfactual:.2f} FTE")

**Interpret (3–5 sentences):** What does the DiD estimate tell us about the minimum wage increase's effect on fast-food employment in New Jersey? Why was this result surprising to many economists at the time? What is the role of Pennsylvania in this analysis?

**Answer:** TODO

## Question 15 — Threats to Identification

List **two** specific threats to the validity of the Card–Krueger DiD estimate. For each threat:
- Describe what would have to happen in the real world for it to be a problem.
- State whether it would bias the estimate **upward** or **downward** relative to the true effect.

**Threat 1:** TODO
- Real-world scenario: TODO
- Direction of bias: TODO

**Threat 2:** TODO
- Real-world scenario: TODO
- Direction of bias: TODO

---
# Part 8: Extension — How Robust is the Estimate? (Code + Interpretation)

## Question 16 — Sensitivity to Noise

Go back to the original simulated dataset. Re-run the simulation **50 times** (each time with a different random seed), saving the $\hat{\beta}_3$ from each run. Then:

**(a)** Plot a histogram of the 50 estimates.  
**(b)** Mark the true ATT (4.0 pp) with a vertical dashed line.  
**(c)** Report the mean and standard deviation of the 50 estimates.

**Clarifying prompt:**

The goal here is not new econometrics. It is repetition.

For each seed:
1. Recreate the simulated panel exactly as in Question 6, but with that seed.
2. Run the same DiD regression as in Question 10.
3. Append the interaction coefficient, `model.params["post_x_treated"]`, to `estimates`.

If helpful, you can copy your code from Question 6 and Question 10 and only change the seed inside the loop.

In [ ]:
estimates = []

for seed in range(50):
    np.random.seed(seed)
    # TODO: Reproduce the data-generation process from Question 6 using this seed,
    #       run the DiD regression, and append beta_3 to `estimates`.
    pass  # replace this

estimates = np.array(estimates)

# TODO: Plot histogram + vertical line for TRUE_ATT

print(f"Mean of estimates : {estimates.mean():.2f}")
print(f"Std of estimates  : {estimates.std():.2f}")

**Interpret (2–3 sentences):** Is the DiD estimator *unbiased* (centered near 4.0)? What does the spread of the distribution tell you about uncertainty when applying DiD to real data?

**Answer:** TODO

---
# Part 9: Reflection

## Question 17 — Explain DiD to a Policymaker

Imagine a policymaker asks: *“What does Difference-in-Differences actually do, and why should I trust it more than a simple before–after comparison?”*

Write a **3–5 sentence** response in plain English. Avoid jargon. Your explanation should mention:
- why a before–after comparison can be misleading,
- what role the control group plays,
- and why parallel trends matters.

**Answer:** TODO

## Question 18 — Synthesis (6–8 sentences)

Reflect on the following:

1. What makes DiD more convincing than a simple before–after comparison?
2. In what situations would you *not* trust a DiD estimate, even if the parallel trends plot looks good?
3. How does the simulation exercise (knowing the true ATT) help you build intuition that you cannot get from real data alone?

**Reflection:** TODO

---
## Final Export

In [ ]:
# Save the cleaned panel to data_clean/
df.to_csv("data_clean/panel_clean.csv", index=False)
print("Clean data saved to data_clean/panel_clean.csv")
print("\nExports folder contents:")
for f in sorted(os.listdir("exports")):
    print(" ", f)

---
## Submission Checklist

Before uploading to Blackboard, confirm:

- [ ] Notebook runs **top → bottom without errors**
- [ ] Every **TODO** replaced with a real answer
- [ ] Both figures saved in `exports/` (parallel trends + comparison)
- [ ] Written answers are in **Markdown cells** (not code comments)
- [ ] Saved as **`.ipynb`** and uploaded to Blackboard Ultra

---
### Further Reading (Optional)
- **Cunningham (2021)**, *Causal Inference: The Mixtape*, Ch. 9 (DiD) — free at [mixtape.scunning.com](https://mixtape.scunning.com)  
- **Card & Krueger (1994)**, "Minimum Wages and Employment," *American Economic Review*  
- **Angrist & Pischke (2009)**, *Mostly Harmless Econometrics*, Ch. 5